# 05 — Tabulares CV + Target Encoding (Roxy)

**Objetivo:** reducir el overfitting del modelo optimizado y mejorar el kappa test.

| Versión anterior | Este notebook |
|---|---|
| Optuna optimiza sobre test split | Optuna optimiza sobre **5-fold CV** |
| Breed1/State como código numérico | **Target Encoding** para Breed1 y State |
| 39 features | 39 + 2 encoded = **41 features** |
| Gap overfitting: 0.2769 | Esperado: < 0.20 |

**Metodología:** reproducimos la técnica robusta del notebook original del profesor
(5-fold CV estratificado + early stopping) pero ahora sobre el FE v3 completo.

## Sección A: Imports y carga de datos

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

BASE_DIR = r'C:/Users/User/Desktop/MCD/Laboratorio de Implementacion II/GitHub/UA_MDM_Labo2'

train_raw = pd.read_csv(os.path.join(BASE_DIR, 'input/train/train.csv'))
sent_df   = pd.read_csv(os.path.join(BASE_DIR, 'input/train_sentiment_features.csv'))
meta_df   = pd.read_csv(os.path.join(BASE_DIR, 'input/train_metadata_features.csv'))

train_raw['desc_length'] = train_raw['Description'].fillna('').apply(len)

df = (train_raw
      .merge(sent_df[['PetID','sentiment_score','sentiment_magnitude','n_sentences']], on='PetID', how='left')
      .merge(meta_df[['PetID','avg_label_score','n_labels','crop_confidence']], on='PetID', how='left')
      .fillna(0))

SEED = 42
train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['AdoptionSpeed'])
print(f'Train: {len(train)} | Test: {len(test)}')


Train: 11994 | Test: 2999


## Sección B: Target Encoding para Breed1 y State

**¿Por qué?** Breed1 tiene ~307 valores distintos y State tiene 15. LightGBM los trata como números enteros,
pero la raza '250' no tiene ninguna relación numérica con la raza '251'.

**Target Encoding** reemplaza cada categoría por la tasa promedio de adopción rápida de esa categoría
calculada sobre el train. Así el modelo recibe información real sobre el comportamiento histórico.

**Cuidado con data leakage:** el encoding se calcula SOLO sobre el fold de train en cada split del CV,
nunca incluyendo el fold de validación ni el test set.

In [2]:
def target_encode(train_df, test_df, col, target='AdoptionSpeed', smoothing=10):
    """
    Target encoding con smoothing para evitar overfitting en categorías poco frecuentes.
    Smoothing: mezcla la media de la categoría con la media global.
    Cuanto menos datos hay de una categoría, más se acerca a la media global.
    """
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    # Formula: (count * category_mean + smoothing * global_mean) / (count + smoothing)
    stats['encoded'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    train_encoded = train_df[col].map(stats['encoded']).fillna(global_mean)
    test_encoded  = test_df[col].map(stats['encoded']).fillna(global_mean)
    return train_encoded, test_encoded

# Aplicar target encoding a Breed1 y State
train = train.copy()
test  = test.copy()

train['Breed1_enc'], test['Breed1_enc'] = target_encode(train, test, 'Breed1')
train['State_enc'],  test['State_enc']  = target_encode(train, test, 'State')

print('Target encoding aplicado:')
print(train[['Breed1','Breed1_enc']].drop_duplicates().sort_values('Breed1_enc', ascending=False).head(10))


Target encoding aplicado:
       Breed1  Breed1_enc
5374      294    2.940244
13895     295    2.916519
14356     279    2.858724
3463      296    2.858724
3958      227    2.781801
12583      97    2.781801
1045        0    2.781801
12375     250    2.769686
2066        1    2.763618
1603       25    2.763618


## Sección B2: FE v4 — RescuerID, estadísticas grupales, edad relativa, NLP básico

Features inspiradas en soluciones top de la competencia (7° puesto):
- **RescuerID aggregates**: historial del rescatista (promedio, cantidad, variabilidad)
- **Stats grupales**: AdoptionSpeed promedio por Breed1, State y Type×State
- **Edad relativa**: Age / mediana de edad para esa raza+tipo (captura 'cachorro viejo')
- **NLP básico**: palabras, palabras únicas, longitud promedio, mayúsculas, signos de emoción

**Leakage prevention:** todas las estadísticas se calculan sobre `train` y se mapean a `test`.

In [3]:
# ── RescuerID count (sin target = sin leakage) ─────────────────────
rescuer_count = train.groupby('RescuerID').size().rename('rescuer_n_pets')
train['rescuer_n_pets'] = train['RescuerID'].map(rescuer_count).fillna(1)
test['rescuer_n_pets']  = test['RescuerID'].map(rescuer_count).fillna(1)

# ── Edad relativa a mediana de Breed1+Type (sin target) ─────────────
age_med_map = train.groupby(['Breed1','Type'])['Age'].median().to_dict()
global_age  = train['Age'].median()
for df_ in [train, test]:
    df_['age_median_bt'] = [age_med_map.get((b,t), global_age)
                             for b,t in zip(df_['Breed1'], df_['Type'])]
    df_['age_rel_breed'] = df_['Age'] / (df_['age_median_bt'] + 1)

# ── NLP básico (sin target) ─────────────────────────────────────────
def nlp_feats(df_):
    desc = df_['Description'].apply(lambda x: '' if (x == 0 or str(x).strip() == '') else str(x))
    df_['word_count']      = desc.apply(lambda x: len(x.split()))
    df_['unique_words']    = desc.apply(lambda x: len(set(x.lower().split())))
    df_['avg_word_len']    = desc.apply(lambda x: round(sum(len(w) for w in x.split()) / max(len(x.split()),1), 2))
    df_['uppercase_ratio'] = desc.apply(lambda x: round(sum(c.isupper() for c in x) / max(len(x),1), 4))
    df_['has_exclamation'] = desc.apply(lambda x: int('!' in x))
    return df_

train = nlp_feats(train)
test  = nlp_feats(test)

# NOTA: breed_speed_mean, state_speed_mean calculados con target
# generan leakage cuando se usan fuera del fold — se omiten.
# Breed1 y State ya tienen target encoding correcto (calculado dentro del fold).

NEW_FEATURES = ['rescuer_n_pets', 'age_rel_breed',
                'word_count', 'unique_words', 'avg_word_len',
                'uppercase_ratio', 'has_exclamation']
print(f'FE v4 — {len(NEW_FEATURES)} features sin leakage:')
print(' | '.join(NEW_FEATURES))


FE v4 — 7 features sin leakage:
rescuer_n_pets | age_rel_breed | word_count | unique_words | avg_word_len | uppercase_ratio | has_exclamation


## Sección C: Feature Engineering v3 + variables encoded

Armamos el set completo de 41 features: 39 del notebook anterior + Breed1_enc + State_enc.

In [4]:
def add_features_v3(df):
    df = df.copy()
    df['HasPhoto']            = (df['PhotoAmt'] > 0).astype(int)
    df['HasVideo']            = (df['VideoAmt'] > 0).astype(int)
    df['IsFree']              = (df['Fee'] == 0).astype(int)
    df['AgeGroup']            = pd.cut(df['Age'], bins=[-1,3,12,48,9999], labels=[0,1,2,3]).astype(int)
    df['HealthScore']         = ((df['Vaccinated']==1).astype(int) +
                                 (df['Dewormed']==1).astype(int) +
                                 (df['Sterilized']==1).astype(int))
    df['IsPureBreed']         = (df['Breed2'] == 0).astype(int)
    df['PhotoPerAnimal']      = df['PhotoAmt'] / df['Quantity'].replace(0,1)
    df['Age_x_PhotoAmt']      = df['Age'] * df['PhotoAmt']
    df['IsPureBreed_x_Age']   = df['IsPureBreed'] * df['AgeGroup']
    df['HealthScore_x_Photo'] = df['HealthScore'] * df['HasPhoto']
    df['IsYoungAndFree']      = ((df['AgeGroup'] <= 1) & (df['IsFree'] == 1)).astype(int)
    df['IsHealthyAndPhoto']   = ((df['HealthScore'] == 3) & (df['HasPhoto'] == 1)).astype(int)
    df['FeePerAnimal']        = df['Fee'] / df['Quantity'].replace(0,1)
    return df

train_fe = add_features_v3(train)
test_fe  = add_features_v3(test)

features_base    = ['Type','Age','Breed1','Breed2','Gender','Color1','Color2','Color3',
                    'MaturitySize','FurLength','Vaccinated','Dewormed','Sterilized',
                    'Health','Quantity','Fee','State','VideoAmt','PhotoAmt']
features_fe1     = ['HasPhoto','HasVideo','IsFree','AgeGroup','HealthScore','IsPureBreed','PhotoPerAnimal']
features_fe2     = ['Age_x_PhotoAmt','IsPureBreed_x_Age','HealthScore_x_Photo',
                    'IsYoungAndFree','IsHealthyAndPhoto','FeePerAnimal']
features_fe3     = ['sentiment_score','sentiment_magnitude','n_sentences',
                    'avg_label_score','n_labels','crop_confidence','desc_length']
features_encoded = ['Breed1_enc', 'State_enc']
features_fe4     = NEW_FEATURES

ALL_FEATURES = features_base + features_fe1 + features_fe2 + features_fe3 + features_encoded + features_fe4

X_train = train_fe[ALL_FEATURES]
X_test  = test_fe[ALL_FEATURES]
y_train = train_fe['AdoptionSpeed']
y_test  = test_fe['AdoptionSpeed']

print(f'Total features: {len(ALL_FEATURES)}')
print(f'  Base: {len(features_base)} | FE1: {len(features_fe1)} | FE2: {len(features_fe2)}')
print(f'  FE3 texto/img: {len(features_fe3)} | Target encoded: {len(features_encoded)} | FE v4: {len(features_fe4)}')


Total features: 48
  Base: 19 | FE1: 7 | FE2: 6
  FE3 texto/img: 7 | Target encoded: 2 | FE v4: 7


## Sección D: Optuna con 5-Fold CV + Early Stopping

Cambiamos la metodología clave respecto al notebook anterior:

- **Antes:** Optuna evaluaba en el test set → el modelo aprendía a optimizar para test → overfitting
- **Ahora:** Optuna evalúa en 5-fold CV sobre train → el test nunca influye en los hiperparámetros

Además usamos **early stopping** dentro de cada fold para que el modelo no entrene de más.

⏳ *Esta sección tarda ~20-30 minutos con 30 trials.*

In [5]:
def lgb_kappa_metric(y_pred, y_true):
    """Métrica custom para early stopping en lgb.train."""
    labels = y_true.get_label()
    preds  = y_pred.reshape(-1, 5).argmax(axis=1)
    kappa  = cohen_kappa_score(labels, preds, weights='quadratic')
    return 'kappa', kappa, True  # True = mayor es mejor

def cv_objective(trial):
    params = {
        'objective':     'multiclass',
        'num_class':     5,
        'verbosity':     -1,
        # Espacio restringido para forzar regularización y reducir overfitting
        'num_leaves':        trial.suggest_int('num_leaves', 2, 64),
        'lambda_l1':         trial.suggest_float('lambda_l1', 0.01, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 0.01, 10.0, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
    }

    skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores    = []
    test_scores  = []
    ensemble_preds = np.zeros((len(X_test), 5))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        lgb_tr  = lgb.Dataset(X_tr,  label=y_tr,  free_raw_data=False)
        lgb_val = lgb.Dataset(X_val, label=y_val, free_raw_data=False)

        model = lgb.train(
            params, lgb_tr,
            num_boost_round=1000,
            valid_sets=[lgb_val],
            callbacks=[lgb.early_stopping(20, verbose=False)],
            feval=lgb_kappa_metric,
        )

        fold_kappa = cohen_kappa_score(
            y_val, model.predict(X_val).argmax(axis=1), weights='quadratic')
        cv_scores.append(fold_kappa)
        ensemble_preds += model.predict(X_test)

    # Test score del ensemble (solo para monitoreo — Optuna NO lo ve)
    test_kappa = cohen_kappa_score(y_test, ensemble_preds.argmax(axis=1), weights='quadratic')
    trial.set_user_attr('test_score', test_kappa)

    return np.mean(cv_scores)  # Optuna optimiza SOLO el CV score

study_cv = optuna.create_study(direction='maximize')
study_cv.optimize(cv_objective, n_trials=30, show_progress_bar=True)

best_cv   = study_cv.best_value
best_test = study_cv.best_trial.user_attrs['test_score']
print('='*60)
print(f'Mejor CV Kappa (5-fold):  {best_cv:.4f}')
print(f'Test Kappa del mejor trial: {best_test:.4f}')
print(f'Params: {study_cv.best_params}')
print('='*60)


  0%|          | 0/30 [00:00<?, ?it/s]

Mejor CV Kappa (5-fold):  0.4086
Test Kappa del mejor trial: 0.3867
Params: {'num_leaves': 51, 'lambda_l1': 0.10207294546882921, 'lambda_l2': 7.578828501649909, 'feature_fraction': 0.5868181778239842, 'bagging_fraction': 0.9754636694026201, 'bagging_freq': 1, 'min_child_samples': 118, 'learning_rate': 0.09296052537790672}


## Sección E: Modelo final y comparativa completa

Entrenamos el modelo final con los mejores hiperparámetros del CV
y armamos el ensemble de los 5 folds para predecir el test.

In [6]:
best_params = study_cv.best_params.copy()
best_params.update({'objective': 'multiclass', 'num_class': 5, 'verbosity': -1})

skf_final      = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ensemble_final = np.zeros((len(X_test), 5))
cv_final       = []

for fold, (tr_idx, val_idx) in enumerate(skf_final.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    lgb_tr  = lgb.Dataset(X_tr,  label=y_tr,  free_raw_data=False)
    lgb_val = lgb.Dataset(X_val, label=y_val, free_raw_data=False)

    m = lgb.train(
        best_params, lgb_tr,
        num_boost_round=1000,
        valid_sets=[lgb_val],
        callbacks=[lgb.early_stopping(20, verbose=False)],
        feval=lgb_kappa_metric,
    )
    cv_final.append(cohen_kappa_score(y_val, m.predict(X_val).argmax(axis=1), weights='quadratic'))
    ensemble_final += m.predict(X_test)

kappa_cv_final   = np.mean(cv_final)
kappa_test_final = cohen_kappa_score(y_test, ensemble_final.argmax(axis=1), weights='quadratic')
kappa_train_check = cohen_kappa_score(y_train,
                       lgb.train(best_params, lgb.Dataset(X_train, label=y_train),
                                 num_boost_round=m.best_iteration)
                       .predict(X_train).argmax(axis=1), weights='quadratic')

print('='*70)
print('  COMPARATIVA FINAL — Evolución completa del proyecto')
print('='*70)
print(f'  Baseline (19 feat)                Test: 0.3133')
print(f'  FE v1    (26 feat)                Test: 0.3231  (+0.0099)')
print(f'  FE v2 + Optuna simple (32 feat)   Test: 0.3371  (+0.0238)')
print(f'  FE v3 + Optuna simple (39 feat)   Test: 0.3595  (+0.0462)')
print(f'  FE v3 + Optuna CV     (41 feat)   Test: {kappa_test_final:.4f}  ({kappa_test_final-0.3133:+.4f})')
print(f'  CV Kappa (5-fold):                      {kappa_cv_final:.4f}')
print(f'  Gap overfitting: Train {kappa_train_check:.4f} - Test {kappa_test_final:.4f} = {kappa_train_check-kappa_test_final:.4f}')
print('='*70)


  COMPARATIVA FINAL — Evolución completa del proyecto
  Baseline (19 feat)                Test: 0.3133
  FE v1    (26 feat)                Test: 0.3231  (+0.0099)
  FE v2 + Optuna simple (32 feat)   Test: 0.3371  (+0.0238)
  FE v3 + Optuna simple (39 feat)   Test: 0.3595  (+0.0462)
  FE v3 + Optuna CV     (41 feat)   Test: 0.3867  (+0.0734)
  CV Kappa (5-fold):                      0.4086
  Gap overfitting: Train 0.6235 - Test 0.3867 = 0.2369


## Sección F: Feature Importance final

In [7]:
import plotly.express as px

# Usar el último modelo del fold para la importancia
fi = pd.DataFrame({'Feature': ALL_FEATURES, 'Importancia': m.feature_importance(importance_type='gain')})
fi = fi.sort_values('Importancia', ascending=False).head(20)

# Clasificar por tipo
tipo_map = {**{f:'Original' for f in features_base},
            **{f:'FE v1' for f in features_fe1},
            **{f:'FE v2' for f in features_fe2},
            **{f:'Texto/Imagen' for f in features_fe3},
            **{f:'Target Encoded' for f in features_encoded}}
fi['Tipo'] = fi['Feature'].map(tipo_map).fillna('Original')

px.bar(fi.sort_values('Importancia'), x='Importancia', y='Feature', orientation='h',
       title='Feature Importance — Top 20 (modelo final CV)',
       color='Tipo', template='plotly_white',
       color_discrete_map={'Original':'#3b82f6','FE v1':'#10b981',
                           'FE v2':'#f59e0b','Texto/Imagen':'#8b5cf6',
                           'Target Encoded':'#ef4444'}).show()
